# 04 — Structured pooling

Structured pooling reduction. Uses `src.features.pooling`.

In [1]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import numpy as np
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils.paths import get_multirocket_features_dir, get_reduced_dir, ensure_dir
from src.features.pooling import structured_pool
from src.features.memmap import open_memmap_read

# Pooling params (match experiments/B1_pooling_mlp and B2_pooling_ft)
N_ORIGINS = 2   # raw + differenced
N_STATS = 4     # PPV, MPV, MIPV, LSPV
POOL = "mean"
print("Pooling: n_origins=%s n_stats=%s pool=%s" % (N_ORIGINS, N_STATS, POOL))

Pooling: n_origins=2 n_stats=4 pool=mean


## What we pool

MultiROCKET outputs one value per **(origin × statistic × kernel)**:

- **Origins (2):** raw series and first-order differenced series (`np.diff(X, 1)`).
- **Statistics (4):** PPV, MPV, MIPV, LSPV (percentage of positive values; mean of positive values; mean of indices of positive values; longest stretch of positive values).
- **Kernels (e.g. 2016):** random convolutional kernels (from `num_kernels=2048` rounded to multiple of 84).

So each sample has shape `(n_origins × n_stats × n_kernels)` = 2×4×2016 = **16,128** features.

**Structured pooling** treats this as a 3D grid `(n_origins, n_stats, n_kernels)` and pools over the first two dimensions (origins and statistics). We use **mean** (set in the first cell as `POOL = "mean"`; the code also supports `"max"`). Result: one value per kernel → 16,128 → **2,016** features per sample.

In [2]:
# Load shape metadata from MultiROCKET output
mr_dir = get_multirocket_features_dir()
shapes_path = mr_dir / "shapes.npz"
if not shapes_path.exists():
    raise FileNotFoundError("Run 02_multirocket.ipynb first to create %s" % shapes_path)
shapes = np.load(shapes_path)
train_shape = tuple(shapes["train"])
test_shape = tuple(shapes["test"])
val_shape = tuple(shapes["val"]) if "val" in shapes else None
print("Train shape:", train_shape)
print("Test shape:", test_shape)
if val_shape:
    print("Val shape:", val_shape)

Train shape: (36120, 16128)
Test shape: (6773, 16128)
Val shape: (2257, 16128)


In [3]:
# Load MultiROCKET features (memmap)
print("Loading train.dat...")
F_train = np.array(open_memmap_read(mr_dir / "train.dat", train_shape), dtype=np.float32)
print("Loading test.dat...")
F_test = np.array(open_memmap_read(mr_dir / "test.dat", test_shape), dtype=np.float32)
if val_shape:
    print("Loading val.dat...")
    F_val = np.array(open_memmap_read(mr_dir / "val.dat", val_shape), dtype=np.float32)
else:
    F_val = None
print("F_train %s F_test %s" % (F_train.shape, F_test.shape))

Loading train.dat...
Loading test.dat...
Loading val.dat...
F_train (36120, 16128) F_test (6773, 16128)


In [4]:
print("Applying structured pooling (see dimension change below)...")

Applying structured pooling (see dimension change below)...


In [5]:
# Structured pooling: reduce (n_samples, n_origins*n_stats*n_kernels) -> (n_samples, n_kernels)
print("Pool operator: %s (over origins and stats)" % POOL)
print("Pooling train...")
P_train = structured_pool(F_train, n_origins=N_ORIGINS, n_stats=N_STATS, pool=POOL)
print("Pooling test...")
P_test = structured_pool(F_test, n_origins=N_ORIGINS, n_stats=N_STATS, pool=POOL)
if F_val is not None:
    print("Pooling val...")
    P_val = structured_pool(F_val, n_origins=N_ORIGINS, n_stats=N_STATS, pool=POOL)
else:
    P_val = None
print("P_train %s P_test %s" % (P_train.shape, P_test.shape))

# Dimension change: before vs after pooling
n_origins, n_stats = N_ORIGINS, N_STATS
n_kernels = F_train.shape[1] // (n_origins * n_stats)
print("\nDimension change (structured pooling):")
print("  Pool: %s over (n_origins, n_stats)" % POOL)
print("  Layout: (n_samples, n_origins × n_stats × n_kernels)  ->  (n_samples, n_kernels)")
print("  With n_origins=%d, n_stats=%d, n_kernels=%d:" % (n_origins, n_stats, n_kernels))
print("  Before: (%s, %d)  ->  After: (%s, %d)" % (F_train.shape[0], F_train.shape[1], P_train.shape[0], P_train.shape[1]))
print("  Reduction: %d -> %d features per sample (factor %d)" % (F_train.shape[1], P_train.shape[1], F_train.shape[1] // P_train.shape[1]))

Pooling train...
Pooling test...
Pooling val...
P_train (36120, 2016) P_test (6773, 2016)

Dimension change (structured pooling):
  Layout: (n_samples, n_origins × n_stats × n_kernels)  ->  (n_samples, n_kernels)
  With n_origins=2, n_stats=4, n_kernels=2016:
  Before: (36120, 16128)  ->  After: (36120, 2016)
  Reduction: 16128 -> 2016 features per sample (factor 8)


In [6]:
# Save pooled features to data/processed/reduced/pooled/ (used by train.py for B1/B2)
out_dir = ensure_dir(get_reduced_dir() / "pooled")
print("Saving to %s..." % out_dir)
np.save(out_dir / "train.npy", P_train.astype(np.float32))
np.save(out_dir / "test.npy", P_test.astype(np.float32))
if P_val is not None:
    np.save(out_dir / "val.npy", P_val.astype(np.float32))
print("Done. Pooled features saved.")

Saving to C:\Projects\DeepLearningProject\data\processed\reduced\pooled...
Done. Pooled features saved.
